# Trustworthy GeoAI — Course Work Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vezarachan/TUM_Course_SVA_TrustGeoAI/blob/main/coursework/TrustGeoAI_CourseWork.ipynb)

**Spatial Visual Analytics · Trustworthy GeoAI**

> **Easiest way to run this:** click the **Open in Colab** badge above — you only need a Google
> account, nothing to install. The first setup cell downloads everything it needs automatically.
>
> **⚠️ FIRST, make your own copy:** in Colab go to **File ▸ Save a copy in Drive**. This opens an
> editable copy in *your* Google Drive — work in that copy so your code, answers, and results are
> saved. (Opening straight from GitHub is read-only: edits there are **not** saved.)

You will investigate **where a GeoAI model can and cannot be trusted**, using *spatial uncertainty*
(prediction intervals from conformal prediction) as the trust signal.

### How to use this notebook  ▶️
1. Run everything once: menu **Run ▸ Run All Cells** (or `Shift+Enter` down the notebook).
2. Then use the **drop-down menus** to choose a dataset, a method, and — most importantly —
   **different visualization methods**.
3. Your job is to **think and compare**: *which visualization best reveals the trustworthiness of
   the model, and what does it tell you?* Write your answers in the 🟨 **YOUR TURN** cells.

> You barely need to write code. The computation is done for you — spend your effort on **choosing
> visualizations**, optionally **building your own view in a few lines**, and **interpreting trust**.
> Plugging in your own idea is how your **unique perspective** comes through (see section ②③).
> The four aspects we grade:
>
> 1. **The scientific question** · 2. **The design of user interface** ·
> 3. **The way of revealing trustworthiness** · 4. **Retrospect the limits of GeoAI** (how to solve the trust problem)

---

## Setup  🟦 PROVIDED — just run it

Imports, the `geocp` method, metadata helpers, the analysis engine, and the **visualization library**.

In [ ]:
import sys, os, re, json, math, io, base64, inspect, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")

# ---- Where to get data/code: works on Google Colab, locally, or a fresh clone ----
REPO, BRANCH = "Vezarachan/TUM_Course_SVA_TrustGeoAI", "main"
RAW = "https://raw.githubusercontent.com/" + REPO + "/" + BRANCH
IN_COLAB = "google.colab" in sys.modules

def _fetch(repo_path, local_path):
    """Download a file from the course GitHub repo if it is not already present."""
    if not os.path.exists(local_path):
        d = os.path.dirname(local_path)
        if d:
            os.makedirs(d, exist_ok=True)
        urllib.request.urlretrieve(RAW + "/" + repo_path, local_path)
    return local_path

# Make geocp importable; download it from GitHub if missing (e.g. on Colab).
HERE = os.getcwd()
ROOT = os.path.dirname(HERE) if os.path.basename(HERE) in ("coursework", "live_demo") else HERE
for p in [HERE, ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
try:
    from geocp import GeoConformalRegressor
except ModuleNotFoundError:
    for _f in ["__init__.py", "core.py", "estimators.py", "results.py", "utils.py", "weights.py"]:
        _fetch("geocp/" + _f, "geocp/" + _f)
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
    from geocp import GeoConformalRegressor

from sklearn.ensemble import RandomForestRegressor   # method: "geocp" or "bayesian"

# Use a local dataset folder if present; otherwise files download on demand.
DATA_DIR = next((d for d in [os.path.join(HERE, "dataset"), os.path.join(ROOT, "dataset")]
                 if os.path.isdir(d)), "dataset")
META_PATH = os.path.join(DATA_DIR, "Metadata.xlsx")

# Datasets you can choose from (each is downloaded the first time it is used).
CSV_FILES = ["US_Health_DIABETES.csv", "US_Health_OBESITY.csv", "US_Health_STROKE.csv",
             "US_Health_CANCER.csv", "US_Health_ARTHRITIS.csv", "US_Health_BPHIGH.csv",
             "US_Health_CASTHMA.csv", "US_Health_DEPRESSION.csv", "US_Hydro_CAMELS.csv",
             "US_Politics_Voting.csv", "US_Forest_FIA.csv", "US_Climate_ERA5_CLIMATE.csv"]

def ensure_csv(name):
    """Local path to a dataset CSV, downloading it from GitHub if needed."""
    return _fetch("coursework/dataset/" + name, os.path.join(DATA_DIR, name))

# Pull the small metadata dictionary so column descriptions work everywhere.
try:
    _fetch("coursework/dataset/metadata.json", os.path.join(DATA_DIR, "metadata.json"))
except Exception:
    pass

# ---- metadata helpers (one sheet per dataset in Metadata.xlsx) ----
def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower().replace(".csv", ""))

_META_JSON = None
_META_WARNED = False
def _load_meta_json():
    global _META_JSON
    if _META_JSON is None:
        p = os.path.join(DATA_DIR, "metadata.json")
        try:
            with open(p, encoding="utf-8") as f:
                _META_JSON = json.load(f)
        except Exception:
            _META_JSON = {}
    return _META_JSON

def _pick_sheet(names, key):
    return ([s for s in names if _norm(s) == key]
            or [s for s in names if key in _norm(s) or _norm(s) in key])

def meta_for(name):
    """Column dictionary for a dataset. Prefers metadata.json (no dependency);
    falls back to Metadata.xlsx (needs openpyxl)."""
    global _META_WARNED
    key = _norm(name)
    mj = _load_meta_json()                      # 1) preferred: bundled JSON
    if mj:
        m = _pick_sheet(list(mj), key)
        if m:
            sh = mj[m[0]]
            return pd.DataFrame(sh["rows"], columns=sh["columns"])
    try:                                        # 2) fallback: read the .xlsx
        xl = pd.ExcelFile(META_PATH)
    except Exception:
        if not mj and not _META_WARNED:
            print("Note: no metadata.json found and openpyxl not installed - column")
            print("      dictionaries are skipped (everything else still works).")
            _META_WARNED = True
        return None
    m = _pick_sheet(xl.sheet_names, key)
    if not m:
        return None
    return xl.parse(m[0]).dropna(how="all", axis=1).dropna(how="all", axis=0)

def dataset_catalog():
    rows = []
    for f in CSV_FILES:
        m = meta_for(f); target = ""
        if m is not None and "Full Name" in m.columns:
            yr = m[m.iloc[:, 0].astype(str).str.strip() == "Y"]
            if len(yr):
                target = str(yr["Full Name"].iloc[0])
        rows.append({"dataset": f, "predicts (Y)": target})
    return pd.DataFrame(rows)

def dataset_info(name):
    df = pd.read_csv(ensure_csv(name))
    print(f"{name}   —   {df.shape[0]} rows x {df.shape[1]} columns")
    m = meta_for(name)
    if m is not None:
        print("\nColumn dictionary (what Y and each X mean):")
        display(m)
    else:
        print("\n(no metadata sheet) columns:", list(df.columns))
    print("\nPreview:"); display(df.head())
    print("\nNumeric summary:"); display(df.describe().round(2))
    return df

print("Setup complete.")

### The analysis engine &amp; visualization library  🟦 PROVIDED — just run it

`run_pipeline()` trains a model and runs the chosen method (results are cached).
`VIZ` is a dictionary of ready-made **visualization methods** — you pick from these.

In [ ]:
# ===== ANALYSIS ENGINE (cached) =====================================
_PIPE = {}

def _numeric_features(df, feat):
    """Make every feature numeric: parse numeric strings, factorize categoricals."""
    cols = []
    for c in feat:
        s = df[c]
        if s.dtype == object:
            sn = pd.to_numeric(s, errors="coerce")
            if sn.notna().mean() > 0.5:            # mostly numeric -> use as number
                cols.append(sn.rename(c))
            else:                                   # categorical text -> integer codes
                cols.append(pd.Series(pd.factorize(s)[0], index=s.index, name=c).astype(float))
        else:
            cols.append(s.astype(float))
    return pd.concat(cols, axis=1)

def run_pipeline(dataset, method="geocp", alpha=0.10, bandwidth=0.3):
    key = (dataset, method, alpha, bandwidth)
    if key in _PIPE:
        return _PIPE[key]
    df = pd.read_csv(ensure_csv(dataset))
    if "Y" not in df.columns:
        raise ValueError(dataset + " has no 'Y' column - pick a dataset listed by the catalog.")
    feat = [c for c in df.columns if c.startswith("X")]
    if {"lon", "lat"}.issubset(df.columns):
        cc = ["lon", "lat"]
    elif {"proj_x", "proj_y"}.issubset(df.columns):
        cc = ["proj_x", "proj_y"]
    elif {"proj_X", "proj_Y"}.issubset(df.columns):
        cc = ["proj_X", "proj_Y"]
    else:
        raise ValueError("No lon/lat or proj_x/proj_y coordinates in " + dataset)

    # Build numeric frames, then drop any row with missing feature/target/coordinate.
    Xdf = _numeric_features(df, feat)
    ydf = pd.to_numeric(df["Y"], errors="coerce")
    cdf = df[cc].apply(pd.to_numeric, errors="coerce")
    keep = Xdf.notna().all(axis=1) & ydf.notna() & cdf.notna().all(axis=1)
    df, Xdf, ydf, cdf = (df[keep].reset_index(drop=True), Xdf[keep].reset_index(drop=True),
                         ydf[keep].reset_index(drop=True), cdf[keep].reset_index(drop=True))
    if len(df) < 50:
        raise ValueError(dataset + " has too few clean rows after dropping missing values.")
    if len(df) > 1800:                              # keep it fast
        samp = df.sample(1800, random_state=0).index
        df, Xdf, ydf, cdf = (df.loc[samp].reset_index(drop=True), Xdf.loc[samp].reset_index(drop=True),
                             ydf.loc[samp].reset_index(drop=True), cdf.loc[samp].reset_index(drop=True))

    X = Xdf.to_numpy(float); y = ydf.to_numpy(float); coords = cdf.to_numpy(float)
    cstd = (coords - coords.mean(0)) / (coords.std(0) + 1e-9)   # +eps guards constant coords
    rng = np.random.default_rng(0); idx = rng.permutation(len(df))
    a, b = int(0.6 * len(df)), int(0.8 * len(df))
    fi, ci, ti = idx[:a], idx[a:b], idx[b:]
    model = RandomForestRegressor(n_estimators=250, random_state=0, n_jobs=-1).fit(X[fi], y[fi])
    reg = GeoConformalRegressor(model.predict, X[ci], y[ci], cstd[ci],
                                bandwidth=bandwidth, miscoverage_level=alpha)
    res = reg.geo_conformalize(X[ti], y[ti], cstd[ti], bayesian=(method == "bayesian"))
    R = pd.DataFrame({"lon": coords[ti, 0], "lat": coords[ti, 1],
                      "pred": res.pred_value, "truth": res.true_value,
                      "lower": res.lower_bound, "upper": res.upper_bound,
                      "uncertainty": res.uncertainty})
    R["width"]   = R.upper - R.lower
    R["error"]   = (R.pred - R.truth).abs()
    R["covered"] = (R.truth >= R.lower) & (R.truth <= R.upper)
    if res.is_bayesian:
        R["posterior_std"] = res.posterior_std
    if "region" in df.columns:
        R["region"] = df["region"].values[ti]
    R.attrs.update(alpha=alpha, coverage=res.coverage, method=method, dataset=dataset)
    summary = (f"{dataset}  |  method={method}  |  coverage {res.coverage:.1%} "
               f"(target {1-alpha:.0%})  |  mean width {res.mean_width_finite:.3f}")
    _PIPE[key] = (R, summary)
    return _PIPE[key]

# ===== STYLE — colors / basemap / point size (set by the explorer controls) =====
STYLE = {"cmap": "auto", "basemap": True, "size": 18}
PALETTES = ["auto", "viridis", "plasma", "magma", "cividis", "coolwarm",
            "YlOrRd", "YlGnBu", "Spectral", "RdYlGn", "Blues", "Reds"]

def _eff_cmap(warn):
    c = STYLE.get("cmap", "auto")
    return c if c and c != "auto" else ("YlOrRd" if warn else "viridis")

# ---- a clean, report-ready matplotlib look for every chart ----
plt.rcParams.update({
    "figure.facecolor": "white", "savefig.facecolor": "white", "savefig.dpi": 130,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlepad": 8,
    "axes.labelcolor": "#444", "axes.edgecolor": "#cccccc", "axes.linewidth": 0.8,
    "xtick.color": "#666", "ytick.color": "#666", "font.size": 10, "legend.frameon": False,
})

# ---- equal-area projection so US maps keep correct proportions (Albers conic) ----
def _albers(lon, lat):
    lon = np.asarray(lon, float); lat = np.asarray(lat, float)
    l0, p0, p1, p2 = np.deg2rad([-96.0, 23.0, 29.5, 45.5])   # CONUS standard parallels
    lam, phi = np.deg2rad(lon), np.deg2rad(lat)
    n = 0.5 * (np.sin(p1) + np.sin(p2))
    C = np.cos(p1) ** 2 + 2 * n * np.sin(p1)
    rho = np.sqrt(np.maximum(C - 2 * n * np.sin(phi), 1e-9)) / n
    rho0 = np.sqrt(max(C - 2 * n * np.sin(p0), 1e-9)) / n
    theta = n * (lam - l0)
    return rho * np.sin(theta), rho0 - rho * np.cos(theta)

def _xy(R):
    """Projected display coordinates: Albers for lon/lat data, as-is if already projected."""
    lon = R["lon"].to_numpy(float); lat = R["lat"].to_numpy(float)
    return _albers(lon, lat) if _is_geographic(R) else (lon, lat)

def _finish_map(ax):
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)

def _frame(ax, xs, ys, margin=0.06):
    """Zoom the map to the data extent (so empty basemap area is cropped)."""
    xs = np.asarray(xs, float); ys = np.asarray(ys, float)
    xs = xs[np.isfinite(xs)]; ys = ys[np.isfinite(ys)]
    if len(xs) < 2:
        return
    dx = (xs.max() - xs.min()) or 1.0; dy = (ys.max() - ys.min()) or 1.0
    ax.set_xlim(xs.min() - margin * dx, xs.max() + margin * dx)
    ax.set_ylim(ys.min() - margin * dy, ys.max() + margin * dy)

def _clean(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.grid(axis="y", color="#eef0f2", zorder=0)

# ---- US-states basemap (drawn only when coordinates look like lon/lat) ----
_STATES = None
def _load_states():
    global _STATES
    if _STATES is None:
        try:
            p = _fetch("coursework/assets/us_states.geojson", os.path.join(DATA_DIR, "us_states.geojson"))
            with open(p, encoding="utf-8") as f:
                _STATES = json.load(f)
        except Exception:
            _STATES = {"features": []}
    return _STATES

def _is_geographic(R):
    return bool(R["lon"].between(-180, 180).all() and R["lat"].between(-90, 90).all())

def _basemap(ax, R):
    if not STYLE.get("basemap", True) or not _is_geographic(R):
        return
    for feat in _load_states().get("features", []):
        geom = feat.get("geometry") or {}
        t = geom.get("type"); coords = geom.get("coordinates") or []
        if t == "Polygon":
            rings = coords
        elif t == "MultiPolygon":
            rings = [r for part in coords for r in part]
        else:
            rings = []
        for ring in rings:
            xs, ys = _albers([c[0] for c in ring], [c[1] for c in ring])
            ax.fill(xs, ys, color="#eef0f3", zorder=0)
            ax.plot(xs, ys, color="#c2c6cc", lw=0.6, zorder=1)

def _newax(ax, w=7, h=5):
    if ax is not None:
        return ax, False
    fig, ax = plt.subplots(figsize=(w, h))
    return ax, True

def _cbar(ax, sc, label):
    cb = ax.figure.colorbar(sc, ax=ax, shrink=.78, pad=.02)
    cb.set_label(label, fontsize=9); cb.ax.tick_params(labelsize=8); cb.outline.set_visible(False)
    return cb

def _scatter_map(ax, R, values, label, warn):
    x, y = _xy(R); v = np.asarray(values, float); m = np.isfinite(v)
    _basemap(ax, R)
    sc = ax.scatter(x[m], y[m], c=v[m], cmap=_eff_cmap(warn),
                    s=STYLE["size"], edgecolor="white", linewidth=.3, zorder=3)
    _cbar(ax, sc, label); _frame(ax, x[m], y[m]); _finish_map(ax)

# ===== VISUALIZATION LIBRARY (each accepts an optional `ax` for dashboards) =====
def _fin(R, col):
    return R[np.isfinite(R[col])]

def v_unc(R, ax=None):
    ax, own = _newax(ax); _scatter_map(ax, R, R["uncertainty"], "interval half-width", True)
    ax.set_title("Uncertainty — where is the model unsure?")
    if own: plt.tight_layout(); plt.show()

def v_pred(R, ax=None):
    ax, own = _newax(ax); _scatter_map(ax, R, R["pred"], "predicted value", False)
    ax.set_title("Prediction — the predicted values")
    if own: plt.tight_layout(); plt.show()

def v_err(R, ax=None):
    ax, own = _newax(ax); _scatter_map(ax, R, R["error"], "|prediction - truth|", True)
    ax.set_title("Error — where is the model actually wrong?")
    if own: plt.tight_layout(); plt.show()

def v_cover_map(R, ax=None):
    ax, own = _newax(ax); _basemap(ax, R)
    x, y = _xy(R); cov = R["covered"].to_numpy()
    ax.scatter(x[cov],  y[cov],  s=max(6, STYLE["size"]-4), c="#d6d9de", label="covered",
               zorder=3, edgecolor="white", linewidth=.2)
    ax.scatter(x[~cov], y[~cov], s=STYLE["size"]+5,         c="#e23b3b", label="missed",
               zorder=4, edgecolor="white", linewidth=.3)
    ax.legend(loc="lower left", fontsize=8)
    ax.set_title("Coverage hit / miss — where do intervals fail?"); _frame(ax, x, y); _finish_map(ax)
    if own: plt.tight_layout(); plt.show()

def v_width_hist(R, ax=None):
    ax, own = _newax(ax, 7, 4.2); d = _fin(R, "width")
    ax.hist(d.width, bins=30, color="#2563eb", alpha=.8)
    ax.axvline(d.width.mean(), color="black", ls="--", label=f"mean {d.width.mean():.2f}")
    ax.set_xlabel("prediction-interval width"); ax.set_ylabel("count")
    ax.set_title("Interval-width histogram"); ax.legend(); _clean(ax)
    if own: plt.tight_layout(); plt.show()

def v_honesty(R, ax=None):
    ax, own = _newax(ax, 6, 5); d = _fin(R, "uncertainty")
    ax.scatter(d.uncertainty, d.error, s=12, alpha=.5)
    lim = max(d.uncertainty.max(), d.error.max())
    ax.plot([0, lim], [0, lim], "--", color="grey", label="y = x")
    ax.set_xlabel("predicted uncertainty"); ax.set_ylabel("actual |error|")
    ax.set_title("Honesty check — claimed vs real error"); ax.legend(); _clean(ax)
    if own: plt.tight_layout(); plt.show()

def v_reliability(R, ax=None):
    ax, own = _newax(ax, 6, 5); d = _fin(R, "uncertainty").sort_values("uncertainty")
    if len(d) < 10:
        ax.text(.5, .5, "too few points", ha="center")
    else:
        groups = np.array_split(d, 10)
        mu = [g.uncertainty.mean() for g in groups]; me = [g.error.mean() for g in groups]
        ax.plot(mu, me, "o-")
        lim = max(max(mu), max(me)); ax.plot([0, lim], [0, lim], "--", color="grey", label="ideal")
        ax.set_xlabel("mean predicted uncertainty"); ax.set_ylabel("mean actual |error|"); ax.legend()
    ax.set_title("Reliability — more uncertainty should mean more error"); _clean(ax)
    if own: plt.tight_layout(); plt.show()

def v_grid(R, ax=None):
    ax, own = _newax(ax); _basemap(ax, R)
    x, y = _xy(R); u = R["uncertainty"].to_numpy(); m = np.isfinite(u)
    hb = ax.hexbin(x[m], y[m], C=u[m], gridsize=18, cmap=_eff_cmap(True),
                   reduce_C_function=np.mean, mincnt=1, zorder=2)
    _cbar(ax, hb, "mean half-width")
    ax.set_title("Spatial grid — mean uncertainty per cell"); _frame(ax, x[m], y[m]); _finish_map(ax)
    if own: plt.tight_layout(); plt.show()

def v_region(R, ax=None):
    ax, own = _newax(ax, 7, 5)
    if "region" not in R.columns:
        ax.text(.5, .5, "no region column in this dataset", ha="center")
    else:
        g = R.groupby("region")["covered"].mean().sort_values()
        ax.barh(g.index.astype(str), g.values * 100, color=plt.cm.RdYlGn(g.values))
        tgt = (1 - R.attrs.get("alpha", .1)) * 100
        ax.axvline(tgt, color="#333", ls="--", lw=1, label=f"target {tgt:.0f}%"); ax.legend(fontsize=8)
        ax.set_xlabel("coverage (%)"); _clean(ax); ax.grid(axis="x", color="#eef0f2")
    ax.set_title("Regional coverage — is the guarantee local too?")
    if own: plt.tight_layout(); plt.show()

def v_meta(R, ax=None):
    ax, own = _newax(ax)
    if "posterior_std" not in R.columns:
        ax.text(.5, .5, 'set method = "bayesian" to see this', ha="center")
        ax.set_title("Bayesian meta-uncertainty")
    else:
        _scatter_map(ax, R, R["posterior_std"], "posterior std", True)
        ax.set_title("Bayesian meta-uncertainty — how unsure is the uncertainty?")
    if own: plt.tight_layout(); plt.show()

VIZ = {
    "Uncertainty map (where is the model unsure?)":  v_unc,
    "Prediction map (the values)":                   v_pred,
    "Absolute-error map (where is it wrong?)":       v_err,
    "Coverage hit/miss map (where intervals fail)":  v_cover_map,
    "Interval-width histogram":                      v_width_hist,
    "Honesty check: uncertainty vs error":           v_honesty,
    "Reliability: binned uncertainty vs error":      v_reliability,
    "Spatial grid: mean uncertainty (hexbin)":       v_grid,
    "Regional coverage bars":                        v_region,
    "Bayesian meta-uncertainty map":                 v_meta,
}

# ---- helpers so YOUR OWN view needs only a line or two (accept `ax` for dashboards) ----
def quickmap(R, values, title="", cmap=None, ax=None):
    """Plot any per-point values on the projected map (basemap + your palette)."""
    ax2, own = _newax(ax)
    eff = cmap or _eff_cmap(False)
    x, y = _xy(R); v = np.asarray(values, float); m = np.isfinite(v)
    _basemap(ax2, R)
    sc = ax2.scatter(x[m], y[m], c=v[m], cmap=eff,
                     s=STYLE["size"], edgecolor="white", linewidth=.3, zorder=3)
    _cbar(ax2, sc, ""); ax2.set_title(title); _frame(ax2, x[m], y[m]); _finish_map(ax2)
    if own: plt.tight_layout(); plt.show()

def quickhist(values, title="", bins=30, color="#2563eb", ax=None):
    """Plot a histogram of any values."""
    ax2, own = _newax(ax, 7, 4)
    v = np.asarray(values, float); v = v[np.isfinite(v)]
    ax2.hist(v, bins=bins, color=color, alpha=.8); ax2.set_title(title)
    if own: plt.tight_layout(); plt.show()

def trust_view(name):
    """Decorator: register YOUR function as a visualization named `name`.
    It appears in the explorer, the dashboard, and the report automatically."""
    def deco(fn):
        VIZ[name] = fn
        return fn
    return deco

def _call_view(fn, R, ax):
    if "ax" in inspect.signature(fn).parameters:
        fn(R, ax=ax)
    else:                       # custom view that makes its own figure
        ax.axis("off"); ax.text(.5, .5, "(your view drew its own figure above)", ha="center", va="center")
        fn(R)

def render_dashboard(R, views, ncols=2, show=True):
    """Render a chosen COMBINATION of views in a grid; returns the Figure."""
    views = [v for v in views if v in VIZ]
    if not views:
        print("Select at least one visualization."); return None
    ncols = max(1, min(int(ncols), len(views))); nrows = math.ceil(len(views) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4.8 * nrows), squeeze=False)
    flat = [a for row in axes for a in row]
    for ax, name in zip(flat, views):
        try:
            _call_view(VIZ[name], R, ax)
        except Exception as e:
            ax.axis("off"); ax.text(.5, .5, "error: " + str(e), ha="center")
    for ax in flat[len(views):]:
        ax.axis("off")
    fig.tight_layout()
    if show: plt.show()
    return fig

def fig_to_base64(fig, dpi=120):
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
    buf.seek(0); return base64.b64encode(buf.read()).decode("ascii")

def focus_region(R, region):
    """Restrict R to one region (sets the visualization range); 'All' keeps everything."""
    if region and str(region).strip().lower() not in ("all", "") and "region" in R.columns:
        sub = R[R["region"].astype(str) == str(region).strip()]
        if len(sub) >= 5:
            sub.attrs.update(getattr(R, "attrs", {}))
            return sub
        print(f"(region '{region}' has too few points - showing all instead)")
    return R

def regions_of(R):
    return sorted(R["region"].astype(str).unique()) if "region" in R.columns else []

def gallery(R, names=None):
    for nm in (names or list(VIZ)):
        print("\n" + "=" * 70 + "\n" + nm); VIZ[nm](R)

print("Engine ready. Visualization methods available:")
for k in VIZ: print("  -", k)

### Browse the available datasets  🟦 PROVIDED

Each dataset and the quantity it predicts. Use it to choose one for your study.

In [ ]:
dataset_catalog()

---
## ① The scientific question  🟨 YOUR TURN

Choose **one dataset** below (set `DATASET`). Run the next cell to read its **column dictionary**
(what `Y` and the `X` features mean) and a data preview. Then write your scientific question.

In [ ]:
# The only setting you must choose for aspect ①:
DATASET = "US_Health_DIABETES.csv"   # change to any name from the catalog above

In [ ]:
df_preview = dataset_info(DATASET)

**① Write your scientific question** (double-click to edit this cell).

> *TODO — e.g. Can we trust the model's predicted diabetes prevalence across US states, and in
> which regions is it least reliable? Why might trust break down there?*

Replace the line above with your own question.

---
## ② Design of the interface  &amp;  ③ Revealing trustworthiness  🟨 YOUR TURN

This is the heart of the course work. The explorer below is a small **dashboard builder** — use the
controls to compose the picture *you* want to tell:

- **dataset / method** — what you study, and how trust is quantified.
- **slots 1–6** — put a chart in each slot (leave a slot `(empty)` to skip it).
  **Slot 1 ★ is your featured / biggest panel.**
- **arrange** — pick a **named layout** (e.g. *Side by side*, *Featured 1 big left + 2,3 right*,
  *2 × 2 grid*). A little **picture updates** to show the shape, and the numbers in it are the slot
  numbers — so you can *see* where each chart will go before you read the result. Choose
  *Equal grid* to fall back to a simple grid where **columns** sets how many panels per row.
- **palette** — choose the color scheme; **US basemap** on/off; **point size**.
- **alpha** (target coverage) and **bandwidth** (how local the method is) — tune the story.

You have full freedom at two levels: (1) combine the ready-made charts, and (2) **build your own
view** (next cell) — it appears in the same *charts* list automatically. As you explore, ask:
*which combination makes the model's (un)trustworthiness clearest, and why?*

### Build your own visualization  🟨 YOUR TURN — your creative space

This is where your **own perspective** takes shape. You are **not** limited to the ready-made charts:
you have the **whole table `R`** and **all of matplotlib** (and Plotly). Compute new quantities,
choose any chart type, combine columns, compare regions or methods — whatever best argues *your* point.

**The only rule:** write `def my_view(R, ax=None): ...` and register it with `@trust_view("a name")`.
It then appears in the dashboard **slots** and in the **report**. Keep the `ax` argument and draw on
it (`ax.scatter(...)`, `ax.plot(...)`, …) so it also works inside the multi-panel dashboard.

**What you can use**

| columns in `R` (one row per test point) | helpers you may call |
|---|---|
| `lon, lat` · `pred, truth` · `lower, upper` | `quickmap(R, values, title, cmap, ax)` — map of any values |
| `uncertainty, width, error` · `covered` | `quickhist(values, title, ax)` — a histogram |
| `region` · `posterior_std` (bayesian) | `_xy(R)`→projected x,y · `_basemap(ax, R)` · `STYLE`, `plt`, pandas |

**Directions to think in** (invent your own!):

- a **derived** trust metric — e.g. `uncertainty / error` → over- vs under-confidence
- a **relationship** — predicted value vs uncertainty; error vs a feature `X*`
- a **ranking** — the least-trustworthy points or regions
- a **comparison** — one region vs another, or `geocp` vs `bayesian`
- anything from raw `plt` — lines, bars, 2-D density, annotations, small multiples…

The cell below has **three worked examples** showing the range. Read them, then use the **scratch
cell** that follows to make something that is yours. Each registered view shows up in the slots above
(re-run the dashboard / interactive map / report after you add one).

In [ ]:
# ★ EXAMPLE A — a DERIVED map: is the model over- or under-confident?
@trust_view("Over/under-confidence (uncertainty / error)")
def ex_ratio(R, ax=None):
    ratio = R["uncertainty"] / (R["error"] + 1e-9)        # >1 cautious, <1 over-confident
    quickmap(R, ratio, "uncertainty / error  (>1 cautious, <1 over-confident)", cmap="coolwarm", ax=ax)

# ★ EXAMPLE B — a RELATIONSHIP (non-spatial): do bigger predictions get wider intervals?
@trust_view("Prediction vs uncertainty (hit/miss)")
def ex_scatter(R, ax=None):
    own = ax is None
    if own:
        _, ax = plt.subplots(figsize=(6, 5))
    colors = R["covered"].map({True: "#cfcfcf", False: "#e23b3b"})
    ax.scatter(R["pred"], R["uncertainty"], c=colors, s=16, edgecolor="white", linewidth=.2)
    ax.set_xlabel("predicted value"); ax.set_ylabel("uncertainty")
    ax.set_title("grey = covered  ·  red = missed"); _clean(ax)
    if own:
        plt.show()

# ★ EXAMPLE C — a RANKING with raw matplotlib: the least-trustworthy points
@trust_view("Top-15 least-trustworthy points")
def ex_rank(R, ax=None):
    own = ax is None
    if own:
        _, ax = plt.subplots(figsize=(6, 5))
    d = R.nlargest(15, "uncertainty")
    labels = d["region"].astype(str).values if "region" in d.columns else [str(i) for i in d.index]
    ax.barh(range(len(d)), d["uncertainty"].values, color="#e23b3b")
    ax.set_yticks(range(len(d))); ax.set_yticklabels(labels, fontsize=8); ax.invert_yaxis()
    ax.set_xlabel("interval half-width"); ax.set_title("most uncertain test points"); _clean(ax)
    if own:
        plt.show()

print("3 example views registered — they now appear in the slot menus above.")

In [ ]:
# ===================================================================
# ★ YOUR SCRATCH SPACE — make a view that is YOURS.
#   You have R (all columns) and ax to draw on. Rename it, change the body,
#   register more with @trust_view(...). Then re-run a dashboard / report cell.
# ===================================================================
@trust_view("My view (rename me)")
def my_view(R, ax=None):
    own = ax is None
    if own:
        _, ax = plt.subplots(figsize=(6, 5))
    # TODO — your idea here. A few starting points (uncomment one, or write your own):
    # quickmap(R, R["error"], "where is the error largest?", ax=ax)
    # quickhist(R["width"], "distribution of interval widths", ax=ax)
    # ax.scatter(R["pred"], R["error"], s=12); ax.set_xlabel("pred"); ax.set_ylabel("|error|"); _clean(ax)
    quickmap(R, R["uncertainty"], "my first view", ax=ax)
    if own:
        plt.show()

print("Your view is registered. Re-run the dashboard, interactive map, or report to use it.")

In [ ]:
# === INTERACTIVE DASHBOARD EXPLORER — use the controls; no coding needed ===
# (Re-run this cell whenever you add or edit a custom view above.)
NONE = "(empty)"

# ---- Friendly named layouts. Slot 1 is always the "featured" (biggest) panel. ----
# Each name maps to a mosaic of slot numbers (a repeated number = that panel SPANS;
# '.' = empty gap). Students just pick a name + see the diagram; no syntax to learn.
LAYOUT_PRESETS = {
    "Equal grid (auto — uses 'columns')":     "",
    "Side by side  ( 1 | 2 )":                "12",
    "Stacked  ( 1 over 2 )":                  "1;2",
    "Three across  ( 1 | 2 | 3 )":            "123",
    "Featured 1 big LEFT  + 2,3 right":       "12;13",
    "Featured 1 big TOP  + 2,3 below":        "11;23",
    "Featured 1 big RIGHT  + 2,3 left":       "21;31",
    "2 x 2 grid  ( 1,2 / 3,4 )":              "12;34",
    "2 x 3 grid  ( 1,2,3 / 4,5,6 )":          "123;456",
}
LAYOUT_DIAGRAMS = {
    "Equal grid (auto — uses 'columns')":
        "┌──┬──┐\n│ 1│ 2│   panels fill in slot order;\n├──┼──┤   'columns' sets how many per row\n│ 3│ 4│\n└──┴──┘",
    "Side by side  ( 1 | 2 )":
        "┌────┬────┐\n│ 1  │ 2  │\n└────┴────┘",
    "Stacked  ( 1 over 2 )":
        "┌─────────┐\n│    1    │\n├─────────┤\n│    2    │\n└─────────┘",
    "Three across  ( 1 | 2 | 3 )":
        "┌───┬───┬───┐\n│ 1 │ 2 │ 3 │\n└───┴───┴───┘",
    "Featured 1 big LEFT  + 2,3 right":
        "┌─────┬───┐\n│     │ 2 │\n│  1  ├───┤\n│     │ 3 │\n└─────┴───┘",
    "Featured 1 big TOP  + 2,3 below":
        "┌─────────┐\n│    1    │\n├────┬────┤\n│ 2  │ 3  │\n└────┴────┘",
    "Featured 1 big RIGHT  + 2,3 left":
        "┌───┬─────┐\n│ 2 │     │\n├───┤  1  │\n│ 3 │     │\n└───┴─────┘",
    "2 x 2 grid  ( 1,2 / 3,4 )":
        "┌───┬───┐\n│ 1 │ 2 │\n├───┼───┤\n│ 3 │ 4 │\n└───┴───┘",
    "2 x 3 grid  ( 1,2,3 / 4,5,6 )":
        "┌──┬──┬──┐\n│ 1│ 2│ 3│\n├──┼──┼──┤\n│ 4│ 5│ 6│\n└──┴──┴──┘",
}

def render_mosaic(R, slot_map, mosaic, show=True, ncols_fallback=2):
    """Place panels by a mosaic of slot numbers (rows split by ';'): a repeated number
    makes that slot SPAN that area, '.' = empty gap. Used by the named layouts above;
    you can also pass your own string, e.g. render_mosaic(R, slot_map, '112;113')."""
    try:
        rows = [list(p.strip().replace(" ", ""))
                for p in re.split(r"[;\n]+", mosaic.strip()) if p.strip()]
        w = max(len(r) for r in rows)
        rows = [r + ["."] * (w - len(r)) for r in rows]          # pad ragged rows
        nrows, ncols = len(rows), w
        fig, axd = plt.subplot_mosaic(rows, figsize=(5.4 * ncols, 4.4 * nrows),
                                      empty_sentinel=".")
    except Exception as e:
        print(f"(could not read layout -> {e};  using an equal grid instead)")
        return render_dashboard(R, list(slot_map.values()) or [list(VIZ)[0]], ncols=ncols_fallback)
    for key, ax in axd.items():
        nm = slot_map.get(str(key))
        if nm and nm in VIZ:
            try:
                _call_view(VIZ[nm], R, ax)
            except Exception as e:
                ax.axis("off"); ax.text(.5, .5, "error: " + str(e), ha="center")
        else:
            ax.axis("off"); ax.text(.5, .5, f"slot {key}\n(empty)", ha="center",
                                    va="center", color="#bbb", fontsize=11)
    fig.tight_layout()
    if show: plt.show()
    return fig

def explore(dataset="US_Health_DIABETES.csv", method="geocp", alpha=0.10, bandwidth=0.30,
            palette="auto", basemap=True, point_size=18, columns=2,
            slot1=NONE, slot2=NONE, slot3=NONE, slot4=NONE, slot5=NONE, slot6=NONE,
            layout="Equal grid (auto — uses 'columns')"):
    STYLE["cmap"] = palette; STYLE["basemap"] = basemap; STYLE["size"] = int(point_size)
    slot_vals = [slot1, slot2, slot3, slot4, slot5, slot6]
    R, summary = run_pipeline(dataset, method, alpha, bandwidth)
    print(summary, "\n")
    mosaic = LAYOUT_PRESETS.get(layout, layout)   # named layout -> mosaic (or use a raw string)
    if mosaic.strip():                            # a shaped layout: slots can span / leave gaps
        slot_map = {str(i): s for i, s in enumerate(slot_vals, 1) if s != NONE}
        render_mosaic(R, slot_map, mosaic, ncols_fallback=columns)
    else:                                         # equal grid in slot order
        views = [s for s in slot_vals if s != NONE]
        render_dashboard(R, views or [list(VIZ)[0]], ncols=columns)

try:
    import ipywidgets as widgets
    L = widgets.Layout
    _ds = {"description_width": "90px"}            # keep long labels readable
    _sl = dict(continuous_update=False, style=_ds)
    _opts = [NONE] + list(VIZ)

    # --- build every control once, then arrange them by group ---
    c = dict(
        dataset    = widgets.Dropdown(options=CSV_FILES, value=DATASET, description="dataset", style=_ds, layout=L(width="320px")),
        method     = widgets.Dropdown(options=["geocp", "bayesian"], value="geocp", description="method", style=_ds, layout=L(width="220px")),
        alpha      = widgets.FloatSlider(value=0.10, min=0.02, max=0.30, step=0.02, description="alpha", readout_format=".2f", layout=L(width="320px"), **_sl),
        bandwidth  = widgets.FloatSlider(value=0.30, min=0.10, max=1.00, step=0.05, description="bandwidth", layout=L(width="320px"), **_sl),
        palette    = widgets.Dropdown(options=PALETTES, value="auto", description="palette", style=_ds, layout=L(width="220px")),
        basemap    = widgets.Checkbox(value=True, description="US basemap", indent=False, layout=L(width="150px")),
        point_size = widgets.IntSlider(value=18, min=4, max=60, step=2, description="point size", layout=L(width="300px"), **_sl),
        columns    = widgets.IntSlider(value=2, min=1, max=3, description="columns", layout=L(width="240px"), **_sl),
        slot1 = widgets.Dropdown(options=_opts, value="Uncertainty map (where is the model unsure?)", description="slot 1 ★", style=_ds, layout=L(width="430px")),
        slot2 = widgets.Dropdown(options=_opts, value="Coverage hit/miss map (where intervals fail)", description="slot 2", style=_ds, layout=L(width="430px")),
        slot3 = widgets.Dropdown(options=_opts, value=NONE, description="slot 3", style=_ds, layout=L(width="430px")),
        slot4 = widgets.Dropdown(options=_opts, value=NONE, description="slot 4", style=_ds, layout=L(width="430px")),
        slot5 = widgets.Dropdown(options=_opts, value=NONE, description="slot 5", style=_ds, layout=L(width="430px")),
        slot6 = widgets.Dropdown(options=_opts, value=NONE, description="slot 6", style=_ds, layout=L(width="430px")),
        layout = widgets.Dropdown(options=list(LAYOUT_PRESETS), value="Equal grid (auto — uses 'columns')",
                                  description="arrange", style=_ds, layout=L(width="430px")),
    )

    out = widgets.interactive_output(explore, c)   # output is separate from the controls

    # live picture of the chosen layout, so students see the shape immediately
    _preview = widgets.HTML()
    def _show_diag(*_):
        art = LAYOUT_DIAGRAMS.get(c["layout"].value, "")
        _preview.value = ("<pre style='margin:2px 0 0 6px;font-family:ui-monospace,Consolas,monospace;"
                          "font-size:12.5px;line-height:1.12;color:#334155;background:#f8fafc;"
                          "border:1px solid #e6e8eb;border-radius:6px;padding:6px 12px;display:inline-block'>"
                          + art + "</pre>")
    c["layout"].observe(_show_diag, names="value"); _show_diag()

    def _hdr(t):
        return widgets.HTML(f"<div style='font-weight:600;color:#1e3a8a;margin:6px 0 2px'>{t}</div>")

    _note = widgets.HTML(
        "<div style='font-size:12px;color:#6b7280;margin:2px 0 2px;line-height:1.5'>"
        "Pick a chart for each slot, then choose how to <b>arrange</b> them. "
        "<b>Slot 1 ★ is the featured (biggest) panel</b>; the numbers in the picture are slot numbers. "
        "Empty slots are skipped.</div>")

    box = L(border="1px solid #e6e8eb", padding="6px 12px 12px", margin="0 0 8px")
    panel = widgets.VBox([
        _hdr("① Data &amp; method"),
        widgets.HBox([c["dataset"], c["method"]]),
        _hdr("② Trust knobs — alpha = target coverage, bandwidth = how local"),
        widgets.HBox([c["alpha"], c["bandwidth"]]),
        _hdr("③ Style"),
        widgets.HBox([c["palette"], c["basemap"], c["point_size"], c["columns"]]),
        _hdr("④ Panels — choose the charts, then arrange them"),
        widgets.GridBox([c["slot1"], c["slot2"], c["slot3"], c["slot4"], c["slot5"], c["slot6"]],
                        layout=L(grid_template_columns="repeat(2, 440px)", grid_gap="2px 16px")),
        _note,
        widgets.HBox([c["layout"], _preview]),
    ], layout=box)

    display(panel, out)
except Exception as e:
    print("(ipywidgets not available — call explore(...) by hand instead)")
    print('e.g.  explore(DATASET, "geocp",')
    print('              slot1="Uncertainty map (where is the model unsure?)",')
    print('              slot2="Coverage hit/miss map (where intervals fail)",')
    print('              slot3="Interval-width histogram",')
    print('              layout="Featured 1 big LEFT  + 2,3 right")\n')
    explore(DATASET, "geocp", slot1="Uncertainty map (where is the model unsure?)",
            slot2="Coverage hit/miss map (where intervals fail)")

### Interactive map — drag, zoom, choose your range  🟦 PROVIDED

The dashboard maps above are static pictures. The map below is **interactive**: **drag to pan**,
**scroll or box-select to zoom**, **hover** a point for its values, and use the **region** box to
focus on one area. This is the easiest way to choose the *visualization range* you want to discuss.

In [ ]:
# === INTERACTIVE MAP — drag / zoom / hover (Plotly). Choose a region to set the range. ===
import html as _htmlmod

def imap(dataset="US_Health_DIABETES.csv", method="geocp",
         color_by="uncertainty (interval width)", palette="auto", region="All"):
    R, summary = run_pipeline(dataset, method)
    print(summary)
    regs = regions_of(R)
    if regs:
        print("regions you can type in the 'region' box:", ", ".join(regs))
    R = focus_region(R, region)
    fields = {"uncertainty (interval width)": ("uncertainty", True),
              "predicted value": ("pred", False),
              "absolute error": ("error", True),
              "posterior std (bayesian)": ("posterior_std", True)}
    col, warn = fields.get(color_by, ("uncertainty", True))
    if col not in R.columns:
        print(f"'{col}' is not available here - using uncertainty."); col, warn = "uncertainty", True
    try:
        import plotly.express as px
    except Exception:
        print("\nplotly is not installed -> showing a static map instead.  (pip install plotly)")
        STYLE["cmap"] = palette; VIZ["Uncertainty map (where is the model unsure?)"](R); return
    scale = {"auto": None, "coolwarm": "RdBu_r"}.get(palette, palette)
    if scale is None:
        scale = "YlOrRd" if warn else "Viridis"
    hover = [c for c in ["pred", "uncertainty", "truth", "region"] if c in R.columns]
    if _is_geographic(R):
        fig = px.scatter_geo(R, lat="lat", lon="lon", color=col, scope="usa",
                             color_continuous_scale=scale, hover_data=hover)
        fig.update_geos(fitbounds="locations", showland=True, landcolor="#f3f4f6",
                        showlakes=False, subunitcolor="#c2c6cc", countrycolor="#c2c6cc")
    else:
        fig = px.scatter(R, x="lon", y="lat", color=col, color_continuous_scale=scale, hover_data=hover)
        fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.update_traces(marker=dict(size=8, line=dict(width=.4, color="white")))
    fig.update_layout(height=560, margin=dict(l=0, r=0, t=46, b=0),
                      title=f"{color_by} — drag to pan, scroll/box to zoom",
                      coloraxis_colorbar=dict(title=""))
    # --- Reliable rendering across environments -------------------------------------
    # On Colab, scripts inserted with display(HTML(...)) via innerHTML do NOT run, so a
    # bare to_html() embed shows up blank. Embedding a *full* self-contained document in
    # an <iframe srcdoc=...> fixes this: srcdoc is parsed as a fresh page, so the Plotly
    # CDN loader and the figure-init script actually execute. Elsewhere fig.show() is best.
    if IN_COLAB:
        from IPython.display import HTML, display
        doc = fig.to_html(include_plotlyjs="cdn", full_html=True,
                          default_width="100%", default_height="560px")
        srcdoc = _htmlmod.escape(doc, quote=True)
        display(HTML(f'<iframe srcdoc="{srcdoc}" width="100%" height="580" '
                     f'style="border:none;" loading="eager"></iframe>'))
    else:
        fig.show()

try:
    import ipywidgets as widgets
    widgets.interact(
        imap,
        dataset=widgets.Dropdown(options=CSV_FILES, value=DATASET, description="dataset"),
        method=widgets.Dropdown(options=["geocp", "bayesian"], value="geocp", description="method"),
        color_by=widgets.Dropdown(options=["uncertainty (interval width)", "predicted value",
                                   "absolute error", "posterior std (bayesian)"], description="color by"),
        palette=widgets.Dropdown(options=PALETTES, value="auto", description="palette"),
        region=widgets.Text(value="All", description="region"),
    )
except Exception as e:
    print("(ipywidgets not available — call imap(...) by hand, e.g. imap(DATASET, region='All'))")
    imap(DATASET)

**② Write-up — interface design.** Explain the **combination** of charts (and positions) you chose:
*why this particular set together, and why does this combination let people **see the risk of the
GeoAI model** more clearly* than a single chart would? What should a viewer notice first? *(3-5 sentences.)*

> *TODO: your answer here.*

**③ Write-up — revealing trustworthiness.** Based on the visualization(s) you chose, **where can the
model be trusted and where not?** Point to a concrete region or pattern, and say what evidence in the
plot supports it. *(3-6 sentences.)*

> *TODO: your answer here.*

---
## ④ Retrospect the limits of GeoAI  🟨 YOUR TURN

Run the pointers below (uncertifiable points, local vs. global coverage). They show where this
GeoAI model's trust breaks down — then propose **how the trust problem could be solved**.

In [ ]:
R, summary = run_pipeline(DATASET, "geocp")
frac_inf = float(np.mean(~np.isfinite(R.uncertainty)))
print(summary)
print(f"Points the method could NOT certify (infinite interval): {frac_inf:.1%}")
if "region" in R.columns:
    print("\nLocal coverage by region (a global guarantee can hide local failure):")
    print(R.groupby("region")["covered"].mean().round(2).sort_values())

**④ Write-up — limits of GeoAI &amp; how to solve the trust problem.** Where and why does this GeoAI
model fail to be trustworthy (e.g. sparse data, spatial extrapolation, local miscalibration)? Then
**propose how the trust problem could be addressed** — e.g. better data or sampling, spatially aware
models, calibration, communicating uncertainty, or keeping a human in the loop. *(4-7 sentences.)*

> *TODO: your answer here.*

---
## 📤 Export your report  🟨 YOUR TURN

Turn your exploration into a **standalone `report.html`** — your unique deliverable. Fill in the
fields, choose the **charts that tell your story**, pick the palette/basemap, and click
**Generate report.html**. On Colab it downloads automatically; locally it is written next to the
notebook. Open it in any browser — it embeds your figures and your four write-ups, no dependencies.

In [ ]:
import html as _html

def build_report(title, author, dataset, method, alpha, bandwidth, palette, basemap, point_size,
                 views, q, interface, findings, limits, region="All", filename="report.html"):
    """Render the chosen panels (in slot order) + narrative into a polished report.html."""
    STYLE["cmap"] = palette; STYLE["basemap"] = basemap; STYLE["size"] = int(point_size)
    R, summary = run_pipeline(dataset, method, alpha, bandwidth)
    R = focus_region(R, region)
    views = [v for v in views if v in VIZ] or [list(VIZ)[0]]
    cards = []
    for name in views:                                  # one polished figure per panel
        fig, ax = plt.subplots(figsize=(7, 5))
        try:
            _call_view(VIZ[name], R, ax)
        except Exception as e:
            ax.axis("off"); ax.text(.5, .5, "error: " + str(e), ha="center")
        fig.tight_layout(); cards.append((name, fig_to_base64(fig))); plt.close(fig)
    esc = _html.escape
    css = "<style>" + (
        ":root{--ink:#1a1a1a;--muted:#6b7280;--line:#e6e8eb;--accent:#2563eb}"
        "*{box-sizing:border-box}body{margin:0;background:#fbfbfc;color:var(--ink);line-height:1.6;"
        "font-family:-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif}"
        "header.rpt{background:linear-gradient(135deg,#1e3a8a,#2563eb);color:#fff;padding:34px 0}"
        "header.rpt h1{margin:0;font-size:26px}header.rpt .by{opacity:.9;font-size:14px;margin-top:6px}"
        ".wrap{max-width:980px;margin:0 auto;padding:0 22px 64px}"
        ".chips{margin:18px 0 4px;display:flex;flex-wrap:wrap;gap:8px}"
        ".chip{background:#eef2ff;color:#1e3a8a;border:1px solid #dbe3ff;border-radius:999px;padding:4px 12px;font-size:12.5px}"
        "h2{font-size:19px;margin:30px 0 10px;border-left:4px solid var(--accent);padding-left:10px}"
        ".figs{display:grid;grid-template-columns:repeat(auto-fit,minmax(380px,1fr));gap:18px}"
        ".card{background:#fff;border:1px solid var(--line);border-radius:12px;padding:12px;box-shadow:0 1px 3px rgba(0,0,0,.05)}"
        ".card img{width:100%;border-radius:8px;display:block}.cap{color:var(--muted);font-size:12.5px;margin-top:8px;text-align:center}"
        "p.story{white-space:pre-wrap;background:#fff;border:1px solid var(--line);border-radius:10px;padding:12px 14px;margin-top:6px}"
        "footer{color:var(--muted);font-size:12px;margin-top:42px;border-top:1px solid var(--line);padding-top:14px}"
    ) + "</style>"
    chips = "".join("<span class='chip'>" + esc(c) + "</span>" for c in [
        "dataset: " + str(dataset), "method: " + str(method), "region: " + str(region),
        f"target coverage: {(1-alpha)*100:.0f}%", f"bandwidth: {bandwidth}", "palette: " + str(palette)])
    figs = "".join("<div class='card'><img alt='" + esc(n) + "' src='data:image/png;base64,"
                   + b + "'><div class='cap'>" + esc(n) + "</div></div>" for n, b in cards)
    def story(t, v):
        return "<h2>" + esc(t) + "</h2><p class='story'>" + (esc(str(v)).strip() or "—") + "</p>"
    narr = (story("① The scientific question", q) + story("② The design of user interface", interface)
            + story("③ The way of revealing trustworthiness", findings)
            + story("④ Retrospect the limits of GeoAI & how to solve the trust problem", limits))
    doc = ("<!doctype html><html lang='en'><head><meta charset='utf-8'>"
           "<meta name='viewport' content='width=device-width,initial-scale=1'>"
           "<title>" + esc(str(title)) + "</title>" + css + "</head><body>"
           "<header class='rpt'><div class='wrap'><h1>" + esc(str(title)) + "</h1>"
           "<div class='by'>" + (esc(str(author)).strip() or "Anonymous")
           + " &middot; Trustworthy GeoAI &middot; Spatial Visual Analytics</div></div></header>"
           "<div class='wrap'><div class='chips'>" + chips + "</div>"
           "<div style='color:#6b7280;font-size:12.5px;margin-bottom:6px'>" + esc(summary) + "</div>"
           "<h2>Visualizations</h2><div class='figs'>" + figs + "</div>" + narr
           + "<footer>Generated from the Trustworthy GeoAI course-work notebook.</footer></div></body></html>")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(doc)
    print("Wrote", filename, "(" + str(len(doc) // 1024) + " KB).")
    try:
        from google.colab import files
        files.download(filename)
    except Exception:
        print("Open / download", filename, "from the file browser (left panel).")
    return filename

try:
    import ipywidgets as widgets
    L = widgets.Layout
    _ds = {"description_width": "90px"}
    _opts = ["(empty)"] + list(VIZ)
    _slot = lambda i, v: widgets.Dropdown(options=_opts, value=v, description=f"slot {i}",
                                          style=_ds, layout=L(width="440px"))
    W = {
        "title":     widgets.Text(value="My Trust Report", description="title", style=_ds, layout=L(width="99%")),
        "author":    widgets.Text(value="", description="your name", style=_ds, layout=L(width="99%")),
        "dataset":   widgets.Dropdown(options=CSV_FILES, value=DATASET, description="dataset", style=_ds, layout=L(width="320px")),
        "method":    widgets.Dropdown(options=["geocp", "bayesian"], description="method", style=_ds, layout=L(width="220px")),
        "alpha":     widgets.FloatSlider(value=0.10, min=0.02, max=0.30, step=0.02, description="alpha", readout_format=".2f", style=_ds, layout=L(width="320px")),
        "bandwidth": widgets.FloatSlider(value=0.30, min=0.10, max=1.00, step=0.05, description="bandwidth", style=_ds, layout=L(width="320px")),
        "palette":   widgets.Dropdown(options=PALETTES, value="auto", description="palette", style=_ds, layout=L(width="220px")),
        "basemap":   widgets.Checkbox(value=True, description="US basemap", indent=False, layout=L(width="150px")),
        "point_size":widgets.IntSlider(value=18, min=4, max=60, step=2, description="point size", style=_ds, layout=L(width="300px")),
        "region":    widgets.Text(value="All", description="region", style=_ds, layout=L(width="320px")),
        "slot1":     _slot(1, "Uncertainty map (where is the model unsure?)"),
        "slot2":     _slot(2, "Coverage hit/miss map (where intervals fail)"),
        "slot3":     _slot(3, "Regional coverage bars"),
        "slot4":     _slot(4, "(empty)"),
        "q":         widgets.Textarea(description="① question",  style=_ds, layout=L(width="99%", height="60px")),
        "interface": widgets.Textarea(description="② interface", style=_ds, layout=L(width="99%", height="60px")),
        "findings":  widgets.Textarea(description="③ findings",  style=_ds, layout=L(width="99%", height="80px")),
        "limits":    widgets.Textarea(description="④ limits",    style=_ds, layout=L(width="99%", height="80px")),
    }
    _btn = widgets.Button(description="Generate report.html", button_style="primary", icon="download")
    _out = widgets.Output()
    def _go(_):
        with _out:
            _out.clear_output()
            views = [W[s].value for s in ("slot1", "slot2", "slot3", "slot4") if W[s].value != "(empty)"]
            build_report(W["title"].value, W["author"].value, W["dataset"].value, W["method"].value,
                         W["alpha"].value, W["bandwidth"].value, W["palette"].value, W["basemap"].value,
                         W["point_size"].value, views,
                         W["q"].value, W["interface"].value, W["findings"].value, W["limits"].value,
                         region=W["region"].value)
    _btn.on_click(_go)

    def _hdr(t):
        return widgets.HTML(f"<div style='font-weight:600;color:#1e3a8a;margin:8px 0 2px'>{t}</div>")

    box = L(border="1px solid #e6e8eb", padding="6px 12px 12px", margin="0 0 8px")
    form = widgets.VBox([
        _hdr("Report header"),
        W["title"], W["author"],
        _hdr("Data, method &amp; style"),
        widgets.HBox([W["dataset"], W["method"], W["region"]]),
        widgets.HBox([W["alpha"], W["bandwidth"]]),
        widgets.HBox([W["palette"], W["basemap"], W["point_size"]]),
        _hdr("Panels — the charts that tell your story (in order)"),
        widgets.GridBox([W["slot1"], W["slot2"], W["slot3"], W["slot4"]],
                        layout=L(grid_template_columns="repeat(2, 450px)", grid_gap="2px 16px")),
        _hdr("Your four write-ups"),
        W["q"], W["interface"], W["findings"], W["limits"],
    ], layout=box)
    display(form, _btn, _out)
except Exception as e:
    print("ipywidgets not available — call build_report(...) directly, e.g.:")
    print('build_report("My Report", "Your Name", "US_Health_DIABETES.csv", "geocp", 0.10, 0.30,')
    print('   "auto", True, 18,')
    print('   ["Uncertainty map (where is the model unsure?)", "Coverage hit/miss map (where intervals fail)"],')
    print('   "my question", "my interface rationale", "my trust findings", "my reflection on limits")')

---
## Submission checklist

For each aspect, make sure your notebook contains:

- **①** your dataset choice + a clear scientific question
- **②** the visualization method(s) you chose + why (interface design)
- **③** an interpretation of where the model can/cannot be trusted, citing the plot
- **④** the limits of GeoAI here + your proposal for how to solve the trust problem
- **export** your `report.html` (the *Export your report* section) and submit it together with the notebook

*Tip:* switching `method` to `bayesian`, changing the **palette/basemap**, or comparing two datasets
often changes the trust picture — great material for ③ and ④.